In [ ]:
"""
adding notagen corpus
"""

from music21 import *
notagencorpus = corpus.corpora.LocalCorpus("notagen")
notagencorpus.addPath(r"~\notagen-pieces\scores\Baroque_BachJohannSebastian_Choral")
notagencorpus.save()
for path in notagencorpus.getPaths():
    score = corpus.parse(path) # is that a way to iterate over our notagen corpus?

In [ ]:
"""
1. Keys:
Algorithms: determine key of chorale by applying music21's key finding algorithm.
- count how often each key occurs, e.g.
bach_keys = { "C": 23, "C#": 1. "D": 12, ..."c": 32, "c#": 2, ...}
notagen_keys = { "C": 58, "C#": 1. "D": 6, ..."c": 14, "c#": 2, ...}

2. Time signatures:
Algorithm: take first time signature
bach_tss = { "4/4": 23, "3/4": 1. "5/8": 0, ...}
notagen_tss = { "4/4": 58, "3/4": 1. "5/8": 1, ...}

3. Melodies:
Algorithm:
  - transpose everything to C major / a minor (Bach AND Notagen)
  - take only 10 first notes of Soprano

4. Dominant seventh chords
...

5. Parallel Fiths (Google search)

6. Dv / Neapolitaner
...

7. Auftakte / Volltakte
"""

In [97]:
"""
J:
1. Keys:
Algorithms: determine key of chorale by applying music21's key finding algorithm.
- count how often each key occurs, e.g.
bach_keys = { "C": 23, "C#": 1. "D": 12, ..."c": 32, "c#": 2, ...}
notagen_keys = { "C": 58, "C#": 1. "D": 6, ..."c": 14, "c#": 2, ...}
"""
import music21 as m21
from collections import Counter
keys_BACH = []
keys_notagen = []

# BACHkeys
bach_chorales = m21.corpus.chorales.Iterator()

for c in bach_chorales:
    chorales_key = c.analyze("key")
    keys_BACH.append(f"{chorales_key.tonic.name} {chorales_key.mode}")
keys_total = Counter(keys_BACH)
bach_keys = sorted(keys_total.items()) # this casts keys_total to a list!

# NOTAGENkeys
for path in notagencorpus.getPaths():
    score = corpus.parse(path)
    ai_keys = score.analyze("key")
    keys_notagen.append(f"{ai_keys.tonic.name} {ai_keys.mode}")
ai_keys_total = Counter(keys_notagen)
ai_keys = sorted(ai_keys_total.items())

# output
print("BACH KEYS:")
print(*bach_keys, sep="\n")
print("-----------------")
print("NOTAGEN KEYS:")
print(*ai_keys, sep="\n")



BACH KEYS:
('A major', 32)
('A minor', 45)
('A- major', 1)
('B major', 1)
('B minor', 28)
('B- major', 25)
('B- minor', 2)
('C major', 17)
('C minor', 8)
('D major', 25)
('D minor', 26)
('E major', 8)
('E minor', 15)
('E- major', 10)
('F major', 24)
('F minor', 2)
('F# minor', 6)
('G major', 52)
('G minor', 44)
-----------------
NOTAGEN KEYS:
('A major', 3)
('A minor', 6)
('A- major', 1)
('B minor', 22)
('B- major', 2)
('C major', 4)
('D major', 6)
('D minor', 4)
('E minor', 15)
('F major', 7)
('F minor', 1)
('G major', 19)
('G minor', 10)


In [98]:
"""
2. Time signatures:
Algorithm: take first time signature
bach_tss = { "4/4": 23, "3/4": 1. "5/8": 0, ...}
notagen_tss = { "4/4": 58, "3/4": 1. "5/8": 1, ...}
"""
""" NOTAGEN CHORALES; Iteration doesn't work
from collections import Counter
notagen_tss_list = []
notagen_chorales = m21.corpus.corpora.LocalCorpus("ai-bach").Iterator() #?
for piece in notagen_chorales:
  time_sig = piece.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
  notagen_tss_list.append(time_sig.ratioString)
notagen_tss = Counter(notagen_tss_list)
print(notagen_tss)
"""
from collections import Counter


# BACH

bach_tss_list = []
bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    time_sig = piece.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
    bach_tss_list.append(time_sig.ratioString)
bach_tss = Counter(bach_tss_list)
print(f"BACH TIME SIGNATURES: {bach_tss}")

# NOTAGEN
notagen_tss_list = []
for path in notagencorpus.getPaths():
    score = corpus.parse(path)
    time_sig = score.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
    notagen_tss_list.append(time_sig.ratioString)
notagen_tss_list = Counter(notagen_tss_list)
print(f"NOTAGEN TIME SIGNATURES: {notagen_tss_list}")

    






BACH TIME SIGNATURES: Counter({'4/4': 331, '3/4': 38, '3/2': 1, '12/8': 1})
NOTAGEN TIME SIGNATURES: Counter({'4/4': 100})


In [ ]:
"""
3. Melodies:
Algorithm:
  - transpose everything to C major / a minor (Bach AND Notagen)
  - take only 10 first notes of Soprano
"""
import music21 as m21
def extract_snippets(corpus=None):
  if corpus is not None:
    snippets = {}
    rem_snippets = []
    for chorale in corpus:
      try:
        ## transpose to C
        key = chorale.analyze("key")
        if key.mode == "major":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
        elif key.mode == "minor":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))
        else:
          print("error")

        transposed = chorale.transpose(interval)
        for part in chorale.parts:
          if part.partName == "Soprano": # does that work? We should do smell-tests
            id = transposed.metadata.movementName
            soprano = transposed.recurse().notes # <--- we needed recurse instead of flatten! i dont know why, but with flatten the transposed snippets were not right
            # notes = [note.pitch.nameWithOctave for note in soprano.notes][:10]
            notes = [note.pitch.midi for note in soprano.notes][:10]
            snippets[id] = notes
          else:
            rem_snippets[id]
      except:
        # print("except", chorale)
        pass

    return snippets
  else:
    return None

bach_chorales = m21.corpus.chorales.Iterator()
bach_snippets = extract_snippets(corpus=bach_chorales)
bach_snippets

{'bwv269.mxl': [60, 60, 67, 64, 62, 60, 60, 62, 64, 62],
 'bwv347.mxl': [60, 60, 60, 60, 62, 58, 57, 55, 62, 64],
 'bwv153.1.mxl': [71, 72, 71, 69, 76, 76, 74, 72, 71, 74],
 'bwv86.6.mxl': [67, 67, 67, 67, 70, 69, 67, 65, 67, 64],
 'bwv267.mxl': [67, 69, 67, 65, 64, 65, 67, 65, 64, 65],
 'bwv17.7.mxl': [60, 60, 59, 57, 55, 60, 62, 64, 64, 64],
 'bwv40.8.mxl': [76, 76, 74, 72, 71, 69, 69, 69, 72, 74],
 'bwv248.12-2.mxl': [60, 60, 62, 64, 66, 67, 67, 66, 67, 64],
 'bwv38.6.mxl': [71, 64, 71, 72, 71, 69, 67, 69, 71, 71],
 'bwv65.2.mxl': [69, 69, 69, 71, 71, 72, 71, 69, 67, 72],
 'bwv33.6.mxl': [72, 67, 69, 71, 72, 74, 76, 74, 72, 71],
 'bwv184.5.mxl': [72, 72, 71, 69, 67, 72, 74, 76, 74, 76],
 'bwv311.mxl': [71, 72, 71, 69, 71, 72, 74, 76, 74, 72],
 'bwv145.5.mxl': [69, 69, 69, 76, 78, 79, 78, 76, 74, 76],
 'bwv318.mxl': [60, 60, 62, 64, 65, 67, 69, 67, 69, 71],
 'bwv351.mxl': [69, 69, 69, 68, 72, 71, 69, 68, 68, 68],
 'bwv302.mxl': [72, 72, 72, 67, 69, 71, 72, 71, 69, 67],
 'bwv153.5.mxl

In [129]:
def extract_notagen_snippets(corpus=None):
  if corpus is not None:
    snippets = {}
    for chorale in corpus.getPaths():
      chorale = m21.corpus.parse(chorale)
      try:
        ## transpose to C
        key = chorale.analyze("key")
        if key.mode == "major":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
        elif key.mode == "minor":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))
        else:
          print("error")

        transposed = chorale.transpose(interval)
        women_parts = transposed.recurse().getElementsByClass(m21.stream.Part)["Women"] # filter out the men-parts
        soprano = stream.Part() # add a new soprano part
        for v in women_parts.recurse().getElementsByClass(m21.stream.Voice): #filter the Women-Parts by Voices
          if v.id == "1": # Voice 1 is the Soprano we want to extract.
            for n in v.recurse().notes: # appending the notes of Voice 1 to our new Soprano Part
              soprano.append(n)

        notes = [note.pitch.midi for note in soprano.notes]
        snippets[transposed.metadata.movementName] = notes[:10]

      except:
        print("except", chorale)
        pass
    return snippets
notagen_snippets = extract_notagen_snippets(notagencorpus)
notagen_snippets

{'output1.xml': [72, 72, 71, 69, 67, 72, 72, 71, 72, 72],
 'output10.xml': [64, 65, 67, 67, 65, 64, 62, 62, 64, 66],
 'output100.xml': [64, 67, 65, 64, 62, 64, 66, 67, 67, 64],
 'output11.xml': [69, 69, 71, 72, 71, 76, 74, 72, 71, 72],
 'output12.xml': [69, 69, 71, 72, 71, 76, 74, 72, 71, 72],
 'output13.xml': [76, 76, 74, 72, 71, 69, 76, 78, 79, 76],
 'output14.xml': [64, 60, 62, 64, 65, 67, 67, 69, 67, 65],
 'output15.xml': [76, 76, 81, 80, 81, 76, 74, 76, 77, 76],
 'output16.xml': [76, 76, 74, 72, 71, 76, 74, 72, 71, 72],
 'output17.xml': [69, 69, 71, 72, 71, 68, 64, 72, 71, 69],
 'output18.xml': [60, 60, 62, 64, 62, 64, 65, 64, 62, 60],
 'output19.xml': [60, 67, 67, 67, 67, 69, 67, 65, 64, 62],
 'output2.xml': [76, 76, 76, 76, 81, 81, 81, 80, 76, 79],
 'output20.xml': [69, 69, 71, 72, 71, 76, 74, 72, 71, 72],
 'output21.xml': [69, 64, 69, 69, 71, 67, 66, 66, 64, 60],
 'output22.xml': [60, 67, 67, 67, 67, 69, 67, 65, 64, 62],
 'output23.xml': [60, 60, 67, 67, 65, 64, 62, 64, 65, 64]

In [117]:
def extract_notagen_snippets(corpus=None):
    if corpus is None:
        return {}

    snippets = {}

    for path in corpus.getPaths():
        chorale = m21.corpus.parse(path)

        try:
            key = chorale.analyze("key")
            interval = (
                m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
                if key.mode == "major"
                else m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))
            )
            transposed = chorale.transpose(interval)
            women_parts = transposed.recurse().getElementsByClass(m21.stream.Part)["Women"]
            soprano = stream.Part()
            for v in women_parts.recurse().getElementsByClass(m21.stream.Voice):
                if v.id == "1":
                    for n in v.recurse().notes:
                        soprano.append(n)
            soprano.show("text")
         

            notes = [note.pitch.midi for note in soprano.notes]
            snippets[transposed.metadata.movementName] = notes[:10]

        except Exception as e:
            print("Fehler bei:", path, e)

    return snippets
extract_notagen_snippets(notagencorpus)

{0.0} <music21.note.Note C>
{1.0} <music21.note.Note C>
{2.0} <music21.note.Note B>
{2.5} <music21.note.Note A>
{3.0} <music21.note.Note G>
{4.0} <music21.note.Note C>
{5.0} <music21.note.Note C>
{6.0} <music21.note.Note B>
{7.0} <music21.note.Note C>
{8.0} <music21.note.Note C>
{9.0} <music21.note.Note B>
{10.0} <music21.note.Note A>
{11.0} <music21.note.Note D>
{12.5} <music21.note.Note C>
{13.0} <music21.note.Note B>
{15.0} <music21.note.Note A>
{16.0} <music21.note.Note D>
{17.0} <music21.note.Note D>
{18.0} <music21.note.Note D>
{19.0} <music21.note.Note E>
{20.0} <music21.note.Note D>
{21.0} <music21.note.Note C>
{22.0} <music21.note.Note B>
{23.0} <music21.note.Note C>
{24.0} <music21.note.Note C>
{25.0} <music21.note.Note D>
{26.0} <music21.note.Note E>
{27.0} <music21.note.Note D>
{28.0} <music21.note.Note C>
{29.0} <music21.note.Note B>
{31.0} <music21.note.Note A>
{32.0} <music21.note.Note D>
{33.0} <music21.note.Note B>
{34.0} <music21.note.Note C>
{35.0} <music21.note.Note

{'output1.xml': [72, 72, 71, 69, 67, 72, 72, 71, 72, 72],
 'output10.xml': [64, 65, 67, 67, 65, 64, 62, 62, 64, 66],
 'output100.xml': [64, 67, 65, 64, 62, 64, 66, 67, 67, 64],
 'output11.xml': [69, 69, 71, 72, 71, 76, 74, 72, 71, 72],
 'output12.xml': [69, 69, 71, 72, 71, 76, 74, 72, 71, 72],
 'output13.xml': [76, 76, 74, 72, 71, 69, 76, 78, 79, 76],
 'output14.xml': [64, 60, 62, 64, 65, 67, 67, 69, 67, 65],
 'output15.xml': [76, 76, 81, 80, 81, 76, 74, 76, 77, 76],
 'output16.xml': [76, 76, 74, 72, 71, 76, 74, 72, 71, 72],
 'output17.xml': [69, 69, 71, 72, 71, 68, 64, 72, 71, 69],
 'output18.xml': [60, 60, 62, 64, 62, 64, 65, 64, 62, 60],
 'output19.xml': [60, 67, 67, 67, 67, 69, 67, 65, 64, 62],
 'output2.xml': [76, 76, 76, 76, 81, 81, 81, 80, 76, 79],
 'output20.xml': [69, 69, 71, 72, 71, 76, 74, 72, 71, 72],
 'output21.xml': [69, 64, 69, 69, 71, 67, 66, 66, 64, 60],
 'output22.xml': [60, 67, 67, 67, 67, 69, 67, 65, 64, 62],
 'output23.xml': [60, 60, 67, 67, 65, 64, 62, 64, 65, 64]

In [27]:
def extract_notagen_snippets(corpus=None):
    if corpus is None:
        return {}

    snippets = {}

    for path in corpus.getPaths():
        chorale = m21.corpus.parse(path)

        try:
            key = chorale.analyze("key")
            if key.mode == "major":
                interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
            else:
                interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))

            transposed = chorale.transpose(interval)

            women_part = transposed.parts[0]

            voices = women_part.recurse().getElementsByClass(m21.stream.Voice)

            if voices:
                soprano = voices[0]
                print(soprano)
            else:
                soprano = women_part.flat.notes

            notes = [n.pitch.nameWithOctave for n in soprano.notes][:10]
            snippets[transposed.metadata.movementName] = notes

        except Exception as e:
            print("Fehler bei:", path, e)

    return snippets
extract_notagen_snippets(notagencorpus)


<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>
<music21.stream.Voice 1>


{'output1.xml': ['C5'],
 'output10.xml': ['E4', 'F4', 'G4', 'G4'],
 'output100.xml': ['E4'],
 'output11.xml': ['A4'],
 'output12.xml': ['A4'],
 'output13.xml': ['E5', 'E5', 'D5', 'C5'],
 'output14.xml': ['E4'],
 'output15.xml': ['E5'],
 'output16.xml': ['E5'],
 'output17.xml': ['A4'],
 'output18.xml': ['C4'],
 'output19.xml': ['C4'],
 'output2.xml': ['E5'],
 'output20.xml': ['A4'],
 'output21.xml': ['A4', 'E4', 'A4', 'A4'],
 'output22.xml': ['C4'],
 'output23.xml': ['C4'],
 'output24.xml': ['C4'],
 'output25.xml': ['C4', 'C4', 'G4', 'G4'],
 'output26.xml': ['C4'],
 'output27.xml': ['A4'],
 'output28.xml': ['A4', 'A4', 'E5', 'E5', 'F5'],
 'output29.xml': ['G4', 'G4', 'A4', 'G4'],
 'output3.xml': ['E5'],
 'output30.xml': ['A4'],
 'output31.xml': ['E5'],
 'output32.xml': ['E5', 'E5', 'D5', 'C5'],
 'output33.xml': ['C4'],
 'output34.xml': ['A4'],
 'output35.xml': ['C4'],
 'output36.xml': ['E4'],
 'output37.xml': ['C4'],
 'output38.xml': ['E4'],
 'output39.xml': ['C4'],
 'output4.xml': ['A4

In [19]:
"""
4. Dominant seventh chords
"""
import music21 as m21

# NOTAGEN

d7_count_ai = 0
d7_ai_chord_list = [] #do we want/need a list? maybe we could specify what d7 chords occur often?
for path in notagencorpus.getPaths():
    score = corpus.parse(path)
    scoreChords = score.chordify()
    for chord in scoreChords.recurse().getElementsByClass(m21.chord.Chord):
        if chord.isDominantSeventh():
            d7_ai_chord_list.append([chord.beatStr, chord])
            d7_count_ai += 1
        else:
            continue


# BACH
d7_count = 0
d7_bach_chord_list = []

bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    pieceChords = piece.chordify()
    for chord in pieceChords.recurse().getElementsByClass(m21.chord.Chord):
        if chord.isDominantSeventh():
            d7_bach_chord_list.append([chord.beatStr, chord])
            d7_count += 1
        else:
            continue

print(f"Total D7-Chord Count (Bach): {d7_count}")
print(f"Total D7-Chord Count (Notagen): {d7_count_ai}")

KeyboardInterrupt: 

In [ ]:
"""
5. Check for parallel fifths
"""
# figuredBass.checker.parallelFifths?


## The following cells were saved after updating our colab-document, no further use for now

In [ ]:
"""
getting the Measures + Notes for every piece in the Notagen corpus + Average Values
"""
total_notes = 0
total_notes_list = []
total_measures = 0
for i in range(1, 101):
  ai_piece = m21.converter.parse(f"{DATA}/output{i}.xml")
  ai_piece_notes = len(ai_piece.recurse().notes)
  sopr = ai_piece.parts[0]
  ai_piece_measures = sopr.getElementsByClass("Measure")
  total_notes += ai_piece_notes
  total_notes_list.append(ai_piece_notes)
  total_measures += len(ai_piece_measures)
  print(f" Total Notes for Piece {i}: {ai_piece_notes}, Number of Measures: {len(ai_piece_measures)}", end="")
  print("")
print(f"Average Notes per piece: {total_notes / 100:.2f}")
print(f"Average Measures per piece: {total_measures / 100:.2f}")

variance = np.var(total_notes_list)
stand_dev = np.sqrt(variance)

print(f"Die Standardabweichung beträgt {stand_dev:.2f}")

In [ ]:
import numpy as np
"""
J: Getting the Number of Notes and Number of Measures from every original Bach chorale + Average values
"""
bach_chorales = m21.corpus.chorales.Iterator()
total_notes_list = []
total_notes = 0
total_measures = 0
for piece in bach_chorales:
    piece_notes = len(piece.recurse().notes)                                # notes of a piece
    sopr = piece.parts[0]
    piece_measures = sopr.getElementsByClass("Measure")                     # getting the length (measure number) of a piece
    total_notes += piece_notes                                              # sums up notes for all
    total_notes_list.append(piece_notes)
    total_measures += len(piece_measures)                                   # sums up measures for all iterations
    print(f" Total Notes for {piece.metadata.title}: {piece_notes}, Number of Measures: {len(piece_measures)}", end="")
    print("")
print(f"Average Notes per piece: {total_notes / len(bach_chorales):.2f}")
print(f"Average Measures per piece: {total_measures / len(bach_chorales):.2f}")

variance = np.var(total_notes_list)
stand_dev = np.sqrt(variance)
print(f"Die Standardabweichung der Noten beträgt {stand_dev}")